# Week 2 · 메시지와 Tool — 학생지원 문의 에이전트 만들기

Week 1에서는 학생 문의를 State에 담고, Node와 Edge로 `academic`·`general` 경로를 나누는 **고정된 workflow**를 만들었습니다. 이번 주에는 같은 학생지원 사례를 한 단계 확장합니다. `수강신청 정정 기간이 언제예요?`라는 질문을 받으면, 모델이 학사 규정 Tool이 필요한지 판단하고 조회 결과를 읽은 뒤 답하게 할 것입니다.

오늘의 학습 목표:

1. `HumanMessage`, `AIMessage`, `ToolMessage`, `SystemMessage`의 역할을 설명한다.
2. Node의 **update**가 Reducer를 거쳐 State에 합쳐지는 과정을 예측한다.
3. 파이썬 함수의 이름·매개변수·타입 힌트·docstring이 Tool 설명서(schema)가 되는 이유를 설명한다.
4. `AIMessage.tool_calls → ToolNode → ToolMessage`의 요청과 실행을 구분한다.
5. 직접 쓴 Router와 `tools_condition`을 읽고 간단한 ReAct 구조를 만든다.
6. 정상 종료, Tool 실패, 무한 반복과 `recursion_limit`을 구분한다.


## 이번 주 학습 지도

```text
[키 없이 실행] 메시지 역할과 생명주기
       ↓
[키 없이 실행] State update → Reducer → MessagesState
       ↓
[키 없이 실행] 파이썬 함수 → Tool schema
       ↓
[키 없이 실행] 고정 tool_calls → ToolNode → ToolMessage
       ↓
[키 없이 실행] 손으로 쓴 Router → tools_condition
       ↓
[여기서부터 API] 모델이 Tool 요청 생성
       ↓
[API] assistant → tools → assistant ReAct
       ↓
[종합 실습] 예측 → 실행 → 설명 → 빈칸 → 전이 과제
```

초반부는 OpenAI API 키가 없어도 전부 실행됩니다. LLM이 정말 필요한 시점에서만 키를 읽습니다.


## 학습 방법과 파이썬 준비물

이 노트북은 함수 `def`, 딕셔너리, 리스트, `if`, `for`, `return` 정도를 알면 시작할 수 있습니다. 낯선 표현은 나오기 직전에 다시 풀이합니다.

| 새 표현 | 읽는 법 |
|---|---|
| `name: str` | `name`에 문자열을 받는다는 타입 힌트 |
| `-> str` | 함수가 문자열을 반환한다는 타입 힌트 |
| `state["messages"][-1]` | `messages` 리스트의 마지막 값 |
| `isinstance(x, ToolMessage)` | `x`가 ToolMessage 종류인지 확인 |
| `Annotated[T, f]` | 기본 타입 `T`에 합치는 규칙 `f`를 메모 |

각 실습은 **예측 → 실행 → 결과 설명 → 작은 수정** 순서로 진행합니다. 정답을 먼저 열지 말고 예상 결과를 한 줄 적어 보세요.


## 2-0. 준비 — 아직 API 키는 사용하지 않습니다

Week 0에서 선택한 가상환경과 Jupyter 커널을 그대로 사용합니다. 아래 셀은 패키지를 다시 설치하지 않고 LangGraph가 준비되었는지만 확인합니다. 오류가 나면 Week 0의 설치 절차를 먼저 완료하세요.


In [ ]:
import importlib.metadata as package_metadata
from pathlib import Path

try:
    langgraph_version = package_metadata.version("langgraph")
except package_metadata.PackageNotFoundError as error:
    raise RuntimeError(
        "LangGraph가 없습니다. Week 0에서 가상환경과 패키지 설치를 완료하세요."
    ) from error

print("✅ LangGraph 버전:", langgraph_version)


In [ ]:
import json
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import (
    AIMessage, HumanMessage, SystemMessage, ToolMessage
)
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

print("✅ 메시지와 그래프 도구 import 완료")


### 출력을 읽는 도우미

LLM의 문장은 매번 조금 달라질 수 있습니다. 그래서 이번 주에는 문장 일치보다 **메시지 종류와 `tool_calls` 구조**를 확인합니다. 아래 함수는 메시지 흐름을 짧게 요약합니다.


In [ ]:
def print_message_trace(messages):
    """메시지 종류·내용·Tool 연결 정보를 한 줄씩 표시합니다."""
    for number, message in enumerate(messages, start=1):
        role = type(message).__name__
        content = str(message.content).replace("\n", " ")[:70]
        content = content or "(텍스트 없음)"
        extra = ""
        if isinstance(message, AIMessage) and message.tool_calls:
            names = [call["name"] for call in message.tool_calls]
            extra = f" | tool_calls={names}"
        if isinstance(message, ToolMessage):
            extra = f" | tool_call_id={message.tool_call_id}"
        print(f"{number}. {role}: {content}{extra}")


## 2-1. 메시지는 `문장 + 역할 + 연결 정보`입니다

문자열 `"안녕"`만으로는 누가 한 말인지 알 수 없습니다. 에이전트는 문장에 **역할(role)**을 붙인 메시지 객체를 사용합니다. 학생지원 센터의 문의 기록으로 비유하면 다음과 같습니다.

| 메시지 | 학생지원 비유 | 주요 내용 |
|---|---|---|
| `SystemMessage` | 상담원 업무 매뉴얼 | 에이전트의 역할·원칙·제한 |
| `HumanMessage` | 학생이 제출한 문의 | 사용자의 질문이나 요청 |
| `AIMessage` | 상담원의 판단/답변 초안 | 자연어 답변 **또는 Tool 실행 요청** |
| `ToolMessage` | 학사 시스템의 조회 회신 | Tool 실행 결과와 요청 ID |

중요: `AIMessage`가 항상 최종 답변인 것은 아닙니다. `tool_calls`가 들어 있다면 아직 **요청서**만 작성된 상태입니다.


In [ ]:
role_examples = [
    SystemMessage(content="모의 규정만 안내하고 확인 경로를 함께 알린다."),
    HumanMessage(content="수강신청 정정 기간이 언제예요?"),
    AIMessage(content="학사 규정을 확인하겠습니다."),
    ToolMessage(
        content="[모의] 포털 공지를 확인하세요.",
        tool_call_id="example-call",
    ),
]

print_message_trace(role_examples)
assert [type(message).__name__ for message in role_examples] == [
    "SystemMessage", "HumanMessage", "AIMessage", "ToolMessage"
]


### 결과 읽기

위 네 메시지는 역할 샘플입니다. 아직 서로 자동 연결되거나 Tool이 실행된 것은 아닙니다. 객체의 종류가 다르기 때문에 그래프는 `AIMessage.tool_calls`와 `ToolMessage.tool_call_id` 같은 구조를 읽고 다음 행동을 결정할 수 있습니다.

<details><summary>이해 확인: SystemMessage는 학생이 반드시 보는 안내문인가요?</summary>

아닙니다. 보통 모델의 역할과 제한을 알리는 내부 지시입니다. 사용자에게 보여 줄 정책 안내는 별도 화면이나 최종 답변으로 제공해야 합니다.
</details>


### Tool을 쓰는 대화의 전체 생명주기

```text
HumanMessage  학생: "수강신청 정정 규정을 알려 줘"
      ↓
AIMessage     모델: lookup_academic_policy(topic="수강신청 정정") 요청
      ↓             ※ 이 시점에는 함수가 아직 실행되지 않음
ToolMessage   ToolNode: 모의 규정 조회 결과
      ↓
AIMessage     모델: 조회 결과를 학생용 문장으로 정리
      ↓
END
```

이 순서를 정확히 읽는 것이 오늘 전체 수업의 뿌리입니다.


## 2-2. State와 update: 공용 상담 기록에 수정 메모를 합치기

Week 1의 State는 학생 이름, 질문, 분류, 답변을 담는 **공용 상담 기록**과 같았습니다. Node는 전체 기록을 새로 쓰지 않고, 자기가 바꾼 칸만 update 딕셔너리로 반환합니다.

```text
기존 State + Node가 반환한 update --(채널별 합치기 규칙)--> 다음 State
```

여기서 **채널(channel)**은 `question`, `category`, `messages`처럼 State의 한 칸을 뜻합니다. update에 없는 칸은 사라지지 않습니다.


In [ ]:
inquiry_state = {
    "student_name": "민지",
    "question": "수강신청 정정 기간이 언제예요?",
}
node_update = {"category": "academic"}

next_state = inquiry_state.copy()
next_state.update(node_update)
print("기존 State:", inquiry_state)
print("Node update:", node_update)
print("다음 State:", next_state)
assert next_state["student_name"] == "민지"
assert next_state["category"] == "academic"


### 결과 읽기

| 칸 | 기존 State | Node update | 다음 State |
|---|---|---|---|
| `student_name` | 민지 | 없음 | 민지 유지 |
| `question` | 수강신청… | 없음 | 수강신청… 유지 |
| `category` | 없음 | academic | academic 추가 |

LangGraph가 실제로는 이 합치기를 대신 수행합니다. State는 DB 같은 영구 저장소라기보다, **현재 그래프 실행의 Node들이 공유하는 작업 기록**으로 먼저 이해하면 좋습니다.


### 예측 1: 같은 칸을 두 번 쓰면?

두 Node가 차례로 `events` 칸에 리스트를 반환합니다.

```text
초기 [] → receive가 ["문의 접수"] 반환 → route가 ["학사 분류"] 반환
```

Reducer가 없다면 최종 `events`에는 몇 개가 남을까요? 실행 전에 예상해 보세요.


In [ ]:
class OverwriteLog(TypedDict):
    events: list[str]


def record_receipt(state):
    return {"events": ["문의 접수"]}


def record_route(state):
    return {"events": ["학사 분류"]}


In [ ]:
overwrite_builder = StateGraph(OverwriteLog)
overwrite_builder.add_node("receive", record_receipt)
overwrite_builder.add_node("route", record_route)
overwrite_builder.add_edge(START, "receive")
overwrite_builder.add_edge("receive", "route")
overwrite_builder.add_edge("route", END)
overwrite_graph = overwrite_builder.compile()

overwrite_result = overwrite_graph.invoke({"events": []})
print(overwrite_result)
assert overwrite_result["events"] == ["학사 분류"]


### 설명: 기본 규칙은 채널 단위 덮어쓰기입니다

`route` Node가 `events`에 새 리스트를 반환했으므로, 앞의 `["문의 접수"]`는 사라지고 `["학사 분류"]`가 남습니다. State 전체가 사라진 것이 아니라 **`events` 칸에만 새 값이 와서** 덮어쓴 것입니다.

<details><summary>정답 확인</summary>

한 개입니다. 마지막 update인 `학사 분류`만 남습니다.
</details>


### Reducer: `기존 값`과 `새 update`를 합치는 함수

누적 기록이 필요하면 해당 채널에 Reducer를 지정합니다. `operator.add`는 파이썬의 `+`를 함수로 표현한 것입니다. 리스트에서는 두 리스트를 이어 붙입니다.

```text
old = ["문의 접수"]
update = ["학사 분류"]
operator.add(old, update) → ["문의 접수", "학사 분류"]
```

`Annotated[list[str], operator.add]`는 “이 칸은 문자열 리스트이고, update는 `operator.add`로 합쳐라”라는 메모입니다. Reducer는 Node를 실행하지 않고 **Node가 반환한 뒤** 값을 합칩니다.


In [ ]:
class AccumulatingLog(TypedDict):
    events: Annotated[list[str], operator.add]


append_builder = StateGraph(AccumulatingLog)
append_builder.add_node("receive", record_receipt)
append_builder.add_node("route", record_route)
append_builder.add_edge(START, "receive")
append_builder.add_edge("receive", "route")
append_builder.add_edge("route", END)
append_graph = append_builder.compile()


In [ ]:
append_result = append_graph.invoke({"events": []})
print("덮어쓰기 결과:", overwrite_result["events"])
print("Reducer 결과:", append_result["events"])

assert append_result["events"] == [
    "문의 접수", "학사 분류"
]


### 결과 해석과 오해 방지

| 시점 | Reducer 없음 | `operator.add` 있음 |
|---|---|---|
| 초기 | `[]` | `[]` |
| receive 후 | `[문의 접수]` | `[문의 접수]` |
| route 후 | `[학사 분류]` | `[문의 접수, 학사 분류]` |

- Reducer가 항상 좋은 것은 아닙니다. `status`처럼 현재 값 하나만 필요한 칸은 덮어쓰기가 자연스럽습니다.
- `operator.add`는 같은 값도 다시 붙입니다. 중복 제거 규칙이 아닙니다.
- Reducer는 State 전체가 아니라 표시한 채널에만 적용됩니다.


### `MessagesState`: 메시지용 Reducer가 준비된 State

메시지는 단순 문자열보다 정보가 많으므로 전용 Reducer인 `add_messages`를 사용합니다. 입문 단계에서는 “새 메시지를 대화 뒤에 합쳐 준다”로 이해해도 충분합니다. `MessagesState`는 개념적으로 다음 채널을 이미 준비한 State입니다.

```python
messages: Annotated[list[메시지], add_messages]
```

그러므로 Node는 전체 대화를 복사하지 않고 **새로 생긴 메시지만 리스트에 담아** 반환하면 됩니다.


In [ ]:
def draft_reception_reply(state: MessagesState):
    reply = AIMessage(
        content="문의를 접수했습니다. 학사 규정을 확인하겠습니다."
    )
    return {"messages": [reply]}


message_builder = StateGraph(MessagesState)
message_builder.add_node("draft_reply", draft_reception_reply)
message_builder.add_edge(START, "draft_reply")
message_builder.add_edge("draft_reply", END)
message_graph = message_builder.compile()


In [ ]:
message_result = message_graph.invoke({
    "messages": [HumanMessage(
        content="수강신청 정정 기간이 언제예요?"
    )]
})

print_message_trace(message_result["messages"])
assert isinstance(message_result["messages"][0], HumanMessage)
assert isinstance(message_result["messages"][1], AIMessage)
assert len(message_result["messages"]) == 2


### State 변화 추적

| 시점 | Node가 본 `messages` | Node update | Reducer 후 State |
|---|---|---|---|
| 시작 | HumanMessage 1개 | - | HumanMessage 1개 |
| `draft_reply` | HumanMessage 1개 | AIMessage 1개 | Human + AI, 총 2개 |

`return {"messages": [reply]}`는 대화를 AIMessage 하나로 교체하라는 뜻이 아닙니다. `MessagesState`의 Reducer가 기존 HumanMessage 뒤에 새 AIMessage를 합칩니다.

<details><summary>이해 확인: AIMessage 하나만 반환해도 HumanMessage가 남는 이유는?</summary>

`MessagesState`의 `messages` 채널에 메시지용 Reducer가 지정되어 있기 때문입니다.
</details>


## 2-3. 파이썬 함수가 Tool이 되는 과정

LLM은 학교 포털 DB를 스스로 읽지 못합니다. 모델의 일반 지식에 학교별 규정을 맡기면 날짜나 조건을 지어낼 수도 있습니다. Tool은 에이전트와 실제 정보원을 연결하는 **관리된 창구**입니다.

수업에서는 실제 학교 규정이 아닌 작은 딕셔너리를 “모의 학사 시스템”으로 사용합니다. 아래 내용을 실제 일정으로 인용하면 안 됩니다.


### 함수 머리를 설명서로 읽기

```python
def lookup_academic_policy(topic: str) -> str:
    """학사 지원용 모의 규정을 주제별로 조회합니다."""
```

- 함수 이름 `lookup_academic_policy` → Tool 이름
- 매개변수 `topic` → 모델이 채워야 하는 인자 이름
- `topic: str` → 인자가 문자열이라는 schema 타입 정보
- docstring → 언제 이 Tool을 쓸지 판단하는 설명
- 함수 본문 → `ToolNode`가 실제로 실행할 코드

타입 힌트는 파이썬의 강제 검사기는 아니지만 Tool 인자 설명서를 만드는 데 큰 도움을 줍니다.


In [ ]:
MOCK_ACADEMIC_POLICIES = {
    "수강신청 정정": (
        "[모의 규정] 정정 기간은 학사 공지에서 확인하고, "
        "기간 내 포털의 수강신청 메뉴에서 변경합니다."
    ),
    "복수전공": (
        "[모의 규정] 신청 자격과 선발 기준은 학과별로 다르므로 "
        "학사 공지와 학과 사무실에서 확인합니다."
    ),
}


def lookup_academic_policy(topic: str) -> str:
    """수강·전공 관련 모의 학사 규정을 주제별로 조회합니다.

    Args:
        topic: 조회할 학사 주제. 예: 수강신청 정정, 복수전공
    """
    return MOCK_ACADEMIC_POLICIES.get(
        topic, f"DEMO_NOT_FOUND: '{topic}' 모의 규정이 없습니다."
    )


In [ ]:
academic_tool_schema = convert_to_openai_tool(
    lookup_academic_policy
)
print(json.dumps(academic_tool_schema, ensure_ascii=False, indent=2))

function_schema = academic_tool_schema["function"]
parameters = function_schema["parameters"]
assert function_schema["name"] == "lookup_academic_policy"
assert parameters["properties"]["topic"]["type"] == "string"
assert "topic" in parameters["required"]
print("✅ 함수에서 Tool schema가 만들어졌습니다.")


### schema 출력 해석

| 파이썬 함수 | Tool schema | 모델이 이용하는 법 |
|---|---|---|
| 함수 이름 | `function.name` | 어느 Tool을 요청할지 선택 |
| docstring | `function.description` | Tool의 용도 판단 |
| `topic` | `parameters.properties.topic` | 필요한 인자 이름 확인 |
| `str` | JSON `string` | 맞는 형식의 인자 생성 |

Tool 설명이 애매하면 모델이 잘못된 Tool이나 인자를 고를 수 있습니다. 실무에서 docstring과 타입 힌트는 단순 주석이 아니라 **에이전트가 읽는 API 계약서**입니다.

<details><summary>예측: `topic: str`에서 `str`을 빼면 무엇이 약해질까요?</summary>

모델에게 제공할 인자 타입 정보가 불명확해집니다. 함수가 실행되는지와 모델이 좋은 인자를 만드는지는 별개의 문제입니다.
</details>


## 2-4. 고정된 Tool 요청으로 생명주기를 먼저 확인하기

LLM을 바로 연결하면 “요청을 누가 만들었는지”와 “함수를 누가 실행했는지”가 한꺼번에 보여 헷갈리기 쉽습니다. 먼저 우리가 손으로 고정된 `AIMessage.tool_calls`를 만듭니다.

### 예측 2

아래 `fixed_request` 객체를 만드는 줄까지만 실행했을 때 `lookup_academic_policy` 함수는 이미 실행됐을까요, 아니면 실행 요청만 저장됐을까요?


In [ ]:
fixed_request = AIMessage(
    content="",
    tool_calls=[{
        "name": "lookup_academic_policy",
        "args": {"topic": "수강신청 정정"},
        "id": "policy-call-001",
        "type": "tool_call",
    }],
)

print("요청 구조:", fixed_request.tool_calls)
assert fixed_request.tool_calls[0]["name"] == "lookup_academic_policy"


`fixed_request` 안에는 Tool 이름, 인자, 요청 ID만 있습니다. 함수 본문은 아직 호출되지 않았습니다. 이제 `ToolNode`가 `name`으로 함수를 찾고 `args`를 펼쳐 실행하게 합니다.


In [ ]:
fixed_tool_builder = StateGraph(MessagesState)
fixed_tool_builder.add_node(
    "tools", ToolNode([lookup_academic_policy])
)
fixed_tool_builder.add_edge(START, "tools")
fixed_tool_builder.add_edge("tools", END)
fixed_tool_graph = fixed_tool_builder.compile()


In [ ]:
fixed_result = fixed_tool_graph.invoke({
    "messages": [fixed_request]
})
tool_reply = fixed_result["messages"][-1]

print_message_trace(fixed_result["messages"])
assert isinstance(fixed_result["messages"][0], AIMessage)
assert isinstance(tool_reply, ToolMessage)
assert tool_reply.tool_call_id == "policy-call-001"
print("✅ 요청 ID와 결과 ID가 연결되었습니다.")


### 결과 해석: 요청과 실행을 분리해 읽기

| 단계 | 담당 | 생성된 것 | 아직 없는 것 |
|---|---|---|---|
| 1 | 우리가 고정 요청 작성 | `AIMessage.tool_calls` | Tool 결과 |
| 2 | `ToolNode` | `ToolMessage` | 학생용 최종 답변 |

`tool_call_id`가 `policy-call-001`로 같은 이유는 결과가 어느 요청에 대한 회신인지 연결하기 위해서입니다. 한 AIMessage가 여러 Tool을 요청해도 각 결과를 찾을 수 있습니다.

<details><summary>정답: fixed_request를 만들었을 때 Tool이 실행됐나요?</summary>

아닙니다. AIMessage에 요청서만 들어갔습니다. `ToolNode`가 실행된 뒤에야 ToolMessage가 생겼습니다.
</details>


In [ ]:
unknown_request = AIMessage(
    content="",
    tool_calls=[{
        "name": "lookup_academic_policy",
        "args": {"topic": "졸업사진"},
        "id": "policy-call-unknown",
        "type": "tool_call",
    }],
)
unknown_result = fixed_tool_graph.invoke({
    "messages": [unknown_request]
})
unknown_reply = unknown_result["messages"][-1]
print(unknown_reply.content)
assert isinstance(unknown_reply, ToolMessage)
assert unknown_reply.tool_call_id == "policy-call-unknown"


### “조회 결과 없음”과 “시스템 오류”는 다릅니다

위 `DEMO_NOT_FOUND`는 함수가 정상 실행된 뒤 “해당 자료가 없다”고 반환한 **업무 결과**입니다. 반면 네트워크 끊김, DB 접속 실패, 잘못된 인자 형식은 **시스템 오류**입니다. 실무 Tool은 두 경우를 구분해 로그를 남기고, 사용자에게 다른 안내를 해야 합니다.


## 2-5. Router를 손으로 쓰면 그래프의 판단이 보입니다

assistant Node가 AIMessage를 반환한 뒤에는 두 경로가 있습니다.

```text
마지막 AIMessage에 tool_calls 있음  → "tools" Node
마지막 AIMessage에 tool_calls 없음  → END
```

Router는 Tool을 실행하지 않습니다. State를 읽고 **다음에 갈 Node 이름**만 반환합니다. Week 1에서 `category`를 보고 `academic_route` 또는 `general_route`를 반환한 함수와 같은 역할입니다.


In [ ]:
def route_after_assistant(state: MessagesState):
    last_message = state["messages"][-1]
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    return END


final_answer = AIMessage(content="학사 공지를 확인해 주세요.")
route_with_call = route_after_assistant({"messages": [fixed_request]})
route_without_call = route_after_assistant({"messages": [final_answer]})

print("Tool 요청이 있을 때:", route_with_call)
print("Tool 요청이 없을 때:", route_without_call)
assert route_with_call == "tools"
assert route_without_call == END


### Router의 입력과 출력

| Router가 읽는 것 | 조건 | 반환값 | 실제 행동 |
|---|---|---|---|
| 마지막 AIMessage | `tool_calls` 있음 | `"tools"` | 그래프가 tools Node로 이동 |
| 마지막 AIMessage | `tool_calls` 없음 | `END` | 그래프 종료 |

`state["messages"][-1]`에서 `-1`은 리스트의 마지막 요소입니다. `and` 앞 조건은 마지막 객체가 AIMessage인지 먼저 확인해 다른 메시지에서 `tool_calls`를 읽다가 오류가 나는 것을 막습니다.


In [ ]:
from langgraph.prebuilt import tools_condition

built_in_with_call = tools_condition({"messages": [fixed_request]})
built_in_without_call = tools_condition({"messages": [final_answer]})
print("tools_condition / 요청 있음:", built_in_with_call)
print("tools_condition / 요청 없음:", built_in_without_call)

assert built_in_with_call == route_with_call
assert built_in_without_call == route_without_call


`tools_condition`은 우리가 직접 쓴 `route_after_assistant`와 같은 표준 판단을 편리하게 제공합니다. 짧은 함수를 먼저 읽었기 때문에 이제 축약 도구를 써도 그래프의 판단이 보이지 않는 “마법”이 아닙니다.

**흔한 오해:** Router가 `"tools"`를 반환했다고 Tool 결과가 즉시 생기는 것은 아닙니다. 그래프 엔진이 tools Node로 이동한 뒤 `ToolNode`가 함수를 실행합니다.


## 2-6. 이제 LLM이 Tool 요청서를 만들게 합니다

여기까지는 API 키 없이 실행했습니다. 지금부터는 우리가 손으로 만든 `fixed_request` 대신 모델이 질문과 Tool schema를 읽고 `tool_calls`를 생성합니다.

API 키가 없으면 아래 셀은 오류를 내지 않고 LLM 실습만 건너뜁니다. 키를 추가한 뒤에는 이 절부터 다시 실행하면 됩니다.


In [ ]:
import os
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)

openai_key = os.environ.get("OPENAI_API_KEY", "").strip()
looks_like_placeholder = openai_key.startswith(("your_", "본인의_"))
HAS_OPENAI_KEY = bool(openai_key and not looks_like_placeholder)
if HAS_OPENAI_KEY:
    print("✅ OPENAI_API_KEY를 안전하게 읽었습니다.")
else:
    print("⏭ .env에 유효한 OPENAI_API_KEY가 없어 LLM 셀을 건너뜁니다.")
    print("   .env.example을 .env로 복사한 뒤 placeholder를 본인 키로 바꾸세요.")


In [ ]:
from langchain_openai import ChatOpenAI

if HAS_OPENAI_KEY:
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        timeout=30,
        max_retries=2,
    )
    print("✅ LLM 객체 준비 완료")
else:
    llm = None


In [ ]:
model_request = None
if llm is None:
    print("⏭ 모델의 Tool 요청 생성 실습을 건너뜁니다.")
else:
    llm_with_required_tool = llm.bind_tools(
        [lookup_academic_policy], tool_choice="any"
    )
    model_request = llm_with_required_tool.invoke([
        SystemMessage(content=(
            "모의 학사 규정을 조회하는 학생지원 조수다. "
            "질문의 핵심 주제로 Tool을 정확히 한 번 호출하라."
        )),
        HumanMessage(content="수강신청 정정 기간이 언제예요?"),
    ])


In [ ]:
if model_request is not None:
    print("모델 텍스트:", repr(model_request.content))
    print("모델이 만든 tool_calls:", model_request.tool_calls)
    assert isinstance(model_request, AIMessage)
    assert model_request.tool_calls
    first_call = model_request.tool_calls[0]
    assert first_call["name"] == "lookup_academic_policy"
    assert "topic" in first_call["args"]
    print("✅ tool_calls 구조로 성공을 확인했습니다.")


### 결과 해석

`bind_tools`는 함수를 실행하지 않고 Tool schema를 모델에게 보여 줍니다. `tool_choice="any"`는 이 첫 실습에서 모델이 자연어로 바로 답하지 않고 제공된 Tool 중 하나를 요청하게 합니다.

모델이 만든 `topic` 표현은 조금씩 다를 수 있습니다. 그래서 정확한 문장 일치를 `assert`하지 않고, AIMessage인지·Tool 요청이 있는지·Tool 이름과 인자 키가 맞는지만 검사했습니다. 이것이 확률적 출력을 테스트하는 기본 방식입니다.


## 2-7. 간단한 ReAct: 요청 → 행동 → 관찰 → 답변

ReAct는 모델의 비공개 생각을 출력하는 기법이 아닙니다. 여기서는 모델이 **행동(Action)**으로 Tool을 요청하고, **관찰(Observation)**로 ToolMessage를 읽은 뒤 다음 행동이나 최종 답을 선택하는 그래프 구조를 뜻합니다.

```text
START → assistant ── tool_calls 있음 ─→ tools ─→ assistant
                    └─ tool_calls 없음 ─→ END
```

이 예제의 종료 장치는 명확합니다. 첫 assistant는 Tool을 한 번 요청하고, ToolMessage를 받은 다음 assistant는 Tool이 연결되지 않은 모델로 답합니다. 따라서 최종 AIMessage에는 `tool_calls`가 없고 Router가 END를 선택합니다.


In [ ]:
support_system = SystemMessage(content=(
    "너는 대학 학생지원 조수다. 모의 규정만 사용한다. "
    "첫 단계에서는 Tool을 한 번 사용하라. topic은 수강신청 정정, "
    "복수전공 중 가장 가까운 하나로 정규화하라. "
    "Tool 결과 뒤에는 추가 Tool 없이 모의 자료임을 밝히고 "
    "최종 일정은 공식 공지에서 확인하라고 답하라."
))

required_policy_llm = (
    llm.bind_tools([lookup_academic_policy], tool_choice="any")
    if llm is not None else None
)


def support_assistant(state: MessagesState):
    last_message = state["messages"][-1]
    model = llm if isinstance(last_message, ToolMessage) else required_policy_llm
    response = model.invoke([support_system] + state["messages"])
    return {"messages": [response]}


In [ ]:
react_builder = StateGraph(MessagesState)
react_builder.add_node("assistant", support_assistant)
react_builder.add_node("tools", ToolNode([lookup_academic_policy]))
react_builder.add_edge(START, "assistant")
react_builder.add_conditional_edges("assistant", tools_condition)
react_builder.add_edge("tools", "assistant")
student_support_agent = react_builder.compile()

print(student_support_agent.get_graph().draw_mermaid())


In [ ]:
react_result = None
if llm is None:
    print("⏭ API 키가 없어 ReAct 실행을 건너뜁니다.")
else:
    react_result = student_support_agent.invoke(
        {"messages": [HumanMessage(
            content="수강신청 정정 기간이 언제예요?"
        )]},
        {"recursion_limit": 6},
    )


In [ ]:
if react_result is not None:
    messages = react_result["messages"]
    print_message_trace(messages)
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    ai_messages = [m for m in messages if isinstance(m, AIMessage)]

    assert isinstance(messages[0], HumanMessage)
    assert tool_messages
    assert any(message.tool_calls for message in ai_messages)
    assert isinstance(messages[-1], AIMessage)
    assert not messages[-1].tool_calls
    print("✅ 요청·실행·최종 답변 구조를 확인했습니다.")


### ReAct 메시지/State 추적표

| 순서 | 실행 Node | 새 update | Reducer 후 `messages` | 다음 경로 |
|---:|---|---|---|---|
| 0 | - | HumanMessage | Human | assistant |
| 1 | assistant | AIMessage(tool_calls) | Human → AI(요청) | tools |
| 2 | tools | ToolMessage | Human → AI(요청) → Tool(결과) | assistant |
| 3 | assistant | AIMessage(최종 답) | Human → AI(요청) → Tool → AI(답) | END |

Node는 매번 새 메시지만 update로 반환합니다. 기존 기록이 남는 이유는 `MessagesState`의 Reducer입니다. 모델이 Tool 결과를 읽을 수 있는 이유도 ToolMessage가 같은 State에 합쳐졌기 때문입니다.


### 종료·실패·`recursion_limit` 멘탈 모델

| 상황 | 무슨 일인가? | 설계 원칙 |
|---|---|---|
| 최종 AIMessage에 `tool_calls` 없음 | Router가 END를 고르는 정상 종료 | 정상 경로를 명시적으로 만든다 |
| Tool이 `DEMO_NOT_FOUND` 반환 | 실행 성공, 업무 자료는 없음 | 공식 확인 경로를 안내한다 |
| DB/네트워크/인자 오류 | Tool 시스템 실패 | 입력 검증, 예외 처리, 로그·재시도 정책을 둔다 |
| assistant가 계속 Tool 요청 | 종료 조건 실패 | State 플래그·호출 횟수·모델 분리로 루프를 끝낸다 |
| `recursion_limit` 초과 | 그래프 엔진의 안전 퓨즈 작동 | 정상 종료로 삼지 말고 루프 원인을 고친다 |

`recursion_limit`은 파이썬 함수의 재귀 횟수가 아니라 한 그래프 실행에서 허용할 최대 graph step 제한입니다. 위 예제의 `6`은 정상 경로에 여유를 주면서 예상하지 못한 반복을 막습니다. 이 예제는 ToolMessage 뒤에 Tool 없는 `llm`을 사용하므로 추가 Tool 요청이 불가능합니다. 이것이 코드에 보이는 종료 조건입니다.


### 여기서 반드시 버려야 할 오해

1. **AIMessage가 생기면 Tool도 실행됐다.** → `tool_calls`는 요청서이고 `ToolNode`가 실행한다.
2. **Router가 Tool을 호출한다.** → Router는 다음 Node 이름만 반환한다.
3. **Node가 메시지 하나를 반환하면 기존 대화가 사라진다.** → `MessagesState`의 Reducer가 합친다.
4. **Tool 결과가 바로 사용자 답변이다.** → 보통 모델이 ToolMessage를 읽고 최종 AIMessage를 만든다.
5. **`recursion_limit`이 루프 설계를 대신한다.** → 안전 퓨즈일 뿐, 정상 종료 조건은 별도로 필요하다.
6. **Tool docstring은 사람만 읽는 주석이다.** → 모델이 Tool을 선택하는 schema 설명에 쓰인다.


## 2-8. 종합 실습 — 예측 → 실행 → 설명 → 빈칸 → 전이

### A. 예측

다음 세 질문에 먼저 답해 보세요.

1. assistant의 AIMessage에 `tool_calls` 하나가 있으면 다음 Node는?
2. ToolNode가 반환한 메시지 종류는?
3. 최종 AIMessage에 `tool_calls`가 없으면 Router의 반환값은?

<details><summary>정답 보기</summary>

1. `tools` Node, 2. `ToolMessage`, 3. `END`입니다.
</details>


### B. 실행 결과를 자신의 말로 설명하기

API 키가 있다면 ReAct 셀의 trace를 보고 문장을 완성하세요.

> 첫 AIMessage는 ______를 요청했고, ______가 함수를 실행해 ToolMessage를 만들었다. 두 번째 AIMessage에는 ______가 없어 그래프가 종료되었다.

<details><summary>예시 답안</summary>

첫 AIMessage는 `lookup_academic_policy`를 요청했고, `ToolNode`가 함수를 실행해 ToolMessage를 만들었다. 두 번째 AIMessage에는 `tool_calls`가 없어 그래프가 종료되었다.
</details>


### C. Router 빈칸 채우기

아래 셀의 `"___"` 두 곳을 고쳐 `fixed_request`와 `final_answer`를 정확히 라우팅하세요. 빈칸 상태에서도 오류 없이 실행되며, 앞의 핵심 그래프에는 영향을 주지 않습니다.


In [ ]:
TOOL_DESTINATION = "___"       # 이 값을 고치세요.
FINISH_DESTINATION = "___"     # 이 값을 고치세요.


def fill_blank_router(state):
    last_message = state["messages"][-1]
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return TOOL_DESTINATION
    return FINISH_DESTINATION


print("요청 있음 →", fill_blank_router({"messages": [fixed_request]}))
print("요청 없음 →", fill_blank_router({"messages": [final_answer]}))
print("목표 출력: tools / __end__")


<details><summary>빈칸 정답</summary>

```python
TOOL_DESTINATION = "tools"
FINISH_DESTINATION = END
```

`END`를 출력하면 내부 문자열인 `__end__`가 보입니다. 고친 뒤 다음 검사도 추가해 보세요.

```python
assert fill_blank_router({"messages": [fixed_request]}) == "tools"
assert fill_blank_router({"messages": [final_answer]}) == END
```
</details>


### D. 전이 과제: Week 1의 `scholarship` 분기를 Tool 에이전트로 확장하기

Week 1 종합 과제에서는 `장학/등록금 → scholarship` 제3분기를 추가했습니다. 이제 고정 답변 대신 모의 장학 규정을 조회하는 Tool을 추가해 보세요. 아래 함수가 시작점입니다.


In [ ]:
MOCK_SCHOLARSHIP_POLICIES = {
    "성적우수": (
        "[모의] 선발 기준과 지급 일정은 장학 공지에서 확인합니다."
    ),
    "국가장학": (
        "[모의] 신청 기간·가구원 동의·서류를 공식 포털에서 확인합니다."
    ),
}


def lookup_scholarship_policy(kind: str) -> str:
    """장학 유형별 모의 안내를 조회합니다.

    Args:
        kind: 장학 유형. 예: 성적우수, 국가장학
    """
    return MOCK_SCHOLARSHIP_POLICIES.get(
        kind, f"DEMO_NOT_FOUND: '{kind}' 장학 안내가 없습니다."
    )


print(lookup_scholarship_policy("성적우수"))


#### 과제 스캐폴드

1. `transfer_tools = [lookup_academic_policy, lookup_scholarship_policy]`를 만듭니다.
2. 수강·전공 문의는 학사 Tool, 장학 문의는 장학 Tool을 고르라고 새 `transfer_system`에 적습니다.
3. 두 Tool을 `llm.bind_tools(...)`와 `ToolNode(...)` 모두에 넣습니다.
4. `국가장학 신청 안내를 찾아 줘`를 입력합니다.
5. `AIMessage(tool_calls) → ToolMessage → AIMessage(최종 답)` 구조를 확인합니다.
6. `교환학생 장학`처럼 모의 DB에 없는 입력도 시험합니다.

단순히 Tool 목록에 추가하는 것으로 끝나지 않습니다. 모델이 두 Tool을 구별하도록 이름·인자·docstring과 SystemMessage의 선택 원칙이 서로 맞아야 합니다.


In [ ]:
# 전이 과제 작업 공간: TODO를 채운 뒤 별도 그래프를 만드세요.
transfer_tools = []  # TODO: 두 Tool 함수를 넣기
transfer_question = "TODO"  # TODO: 장학 문의로 바꾸기

print("현재 Tool 수:", len(transfer_tools))
print("현재 질문:", transfer_question)
print("목표: Tool 2개, 장학 질문 1개")


<details><summary>전이 과제 핵심 정답 예시</summary>

```python
transfer_tools = [lookup_academic_policy, lookup_scholarship_policy]
transfer_question = "국가장학 신청 안내를 찾아 줘."
transfer_system = SystemMessage(content=(
    "수강·전공 문의는 lookup_academic_policy, 장학 문의는 "
    "lookup_scholarship_policy를 사용하라. Tool 결과 뒤에는 최종 답만 하라."
))
transfer_required_llm = llm.bind_tools(transfer_tools, tool_choice="any")

def transfer_assistant(state: MessagesState):
    last_message = state["messages"][-1]
    model = llm if isinstance(last_message, ToolMessage) else transfer_required_llm
    return {"messages": [model.invoke([transfer_system] + state["messages"])]}

transfer_builder = StateGraph(MessagesState)
transfer_builder.add_node("assistant", transfer_assistant)
transfer_builder.add_node("tools", ToolNode(transfer_tools))
transfer_builder.add_edge(START, "assistant")
transfer_builder.add_conditional_edges("assistant", tools_condition)
transfer_builder.add_edge("tools", "assistant")
transfer_agent = transfer_builder.compile()

transfer_result = transfer_agent.invoke(
    {"messages": [HumanMessage(content=transfer_question)]},
    {"recursion_limit": 6},
)
print_message_trace(transfer_result["messages"])
```

LLM 출력은 달라질 수 있습니다. 장학 Tool 선택 여부는 문장이 아니라 `tool_calls[0]["name"]`으로 확인하세요.
</details>


### 전이 과제 루브릭 (10점)

| 항목 | 성공 기준 | 점수 |
|---|---|---:|
| Tool 계약 | 의미 있는 이름, 타입 힌트, 구체적 docstring | 2 |
| Tool 등록 | 모델과 `ToolNode` 모두 같은 두 Tool을 받음 | 2 |
| 라우팅 | Tool 요청은 tools로, 최종 답은 END로 이동 | 2 |
| 구조 검증 | AI(tool_calls) → ToolMessage → AI(최종) 흐름 검사 | 2 |
| 실패 처리 | 자료 없음에 공식 확인 경로 안내 | 1 |
| 설명 | 요청 생성자, 실행자, 종료 조건을 자신의 말로 설명 | 1 |

8점 이상이면 다음 주의 멈춤·재개 흐름을 배울 준비가 된 것입니다. 7점 이하면 먼저 ‘요청서와 실행’, ‘Router와 ToolNode’의 차이를 다시 설명해 보세요.


## 용어집

| 용어 | 이 노트북에서의 뜻 |
|---|---|
| Message | 내용과 역할, 필요한 연결 정보를 담은 대화 객체 |
| State | 현재 그래프 실행의 Node들이 공유하는 작업 기록 |
| channel | State의 각 칸. 예: `messages` |
| update | Node가 State에 반영하라고 반환한 변경분 |
| Reducer | 기존 채널 값과 새 update를 합치는 규칙 |
| `MessagesState` | `messages` 채널과 메시지용 Reducer가 준비된 State |
| Tool | 에이전트가 요청할 수 있는 파이썬 함수/외부 기능 |
| schema | Tool의 이름·용도·인자 형식을 적은 설명서 |
| `tool_calls` | AIMessage 안의 Tool 실행 요청 목록 |
| `ToolNode` | 요청을 읽고 해당 함수를 실행하는 Node |
| `ToolMessage` | Tool 결과와 요청 ID를 담은 메시지 |
| Router | State를 읽고 다음 경로를 반환하는 함수 |
| ReAct | 모델의 Tool 행동과 결과 관찰을 반복하는 실행 구조 |
| termination | 더 실행할 Node가 없어 END로 가는 조건 |
| `recursion_limit` | 예상하지 못한 반복을 막는 최대 graph step 제한 |


## 성취도 체크리스트

아래 항목을 코드를 보지 않고 설명할 수 있는지 확인하세요.

- [ ] Human/AI/Tool/System Message의 역할을 한 문장씩 말할 수 있다.
- [ ] Node update와 전체 State의 차이를 설명할 수 있다.
- [ ] Reducer 없음/있음의 리스트 결과를 예측할 수 있다.
- [ ] `MessagesState`에서 HumanMessage가 남는 이유를 말할 수 있다.
- [ ] 함수 시그니처·타입 힌트·docstring이 schema에 반영되는 위치를 찾을 수 있다.
- [ ] `AIMessage.tool_calls`와 Tool 실행을 구분할 수 있다.
- [ ] `tool_call_id`가 필요한 이유를 말할 수 있다.
- [ ] Router가 실행자가 아니라 경로 선택자임을 설명할 수 있다.
- [ ] ReAct trace에서 요청·Tool 결과·최종 답을 찾을 수 있다.
- [ ] 정상 종료, 자료 없음, 시스템 오류, 반복 한도 초과를 구분할 수 있다.


## Week 3 미리보기

이번 주에는 **한 번의 그래프 실행 안에서** 메시지가 흘러가는 과정에 집중했습니다. Week 3에서는 Checkpointer가 State를 저장하는 이유를 배우고, `thread_id`로 멈춘 실행을 다시 찾아 사람의 승인·거절·수정 후 안전하게 재개하는 Human-in-the-loop 흐름으로 연결합니다.


## 📌 오늘의 정리

- 메시지는 문장뿐 아니라 역할과 Tool 연결 정보를 담는다.
- Node는 전체 State가 아니라 update를 반환하고 Reducer가 채널별로 합친다.
- `MessagesState`는 새 메시지를 기존 대화에 합치는 규칙을 이미 갖고 있다.
- 함수 이름·인자·타입 힌트·docstring은 Tool schema의 재료가 된다.
- `AIMessage.tool_calls`는 요청서이고 `ToolNode`가 실행하며 `ToolMessage`가 결과를 전한다.
- Router는 마지막 AIMessage를 보고 tools 또는 END를 선택한다.
- ReAct에는 정상 종료 조건과 안전 한도가 반드시 있어야 한다.
